## 閃亜鉛鉱構造とウルツ鉱構造のエネルギー差


Open Selectである参考論文に掲載された
二元合金の閃亜鉛鉱構造構造とウルツ鉱構造のエネルギー差を用います。

***説明変数***

- 二元合金の元素説明変数,
IP,EA, Highest_occ state energy Lowest_unocc state energy, s,pの原子半径

***目的変数***

- エネルギー差 dE


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
       'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
       'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
target_name = 'dE'

In [ ]:
# 元素組み合わせとｄEを示します。
df[["A","B","dE"]]

In [ ]:
df.hist(target_name)

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist,squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import copy
from scipy.stats import pearsonr
import numpy as np

def make_linkage(df, descriptor_names, target_name, corr="minus_abs_pearson"):
    labels = copy.deepcopy(descriptor_names)
    labels.append(target_name)
    Xraw = df.loc[:,labels].values
    scaler = StandardScaler()
    X = scaler.fit_transform(Xraw)
    df_tmp = pd.DataFrame(X)
    if corr=="minus_abs_pearson":
        corr = 1- np.abs(df_tmp.corr())
    else:
        raise ValueError("unknown corr={}".format(corr))
    pairdistance = squareform(corr)
    Z = linkage(pairdistance)
    return Z, labels

def show_dendrogram(Z,labels, corr):
    fig, ax = plt.subplots()
    dendrogram(Z,labels=labels,orientation="left", ax=ax)
    ax.set_xlabel(corr)
    fig.tight_layout()
    fig.show()
    
corr="minus_abs_pearson"
Z, labels = make_linkage(df, descriptor_names, target_name, corr)
show_dendrogram(Z,labels, corr)

In [ ]:
import os
import seaborn as sns
from copy import deepcopy

IMAGE_DIR = "image_keep"

imgfile = os.path.join(IMAGE_DIR, "ZBWZ_pairplot.png")
if not os.path.isfile(imgfile):
    alllabels = deepcopy(descriptor_names)
    alllabels.append(target_name)
    img = sns.pairplot(df[alllabels])
    os.makedirs(IMAGE_DIR, exist_ok=True)
    img.savefig(imgfile)
    
from IPython import display
display.Image(imgfile)

更に、参考論文に従い目的変数をうまく説明できる３つの説明変数を作成しています。
これをsymbolic regressionと言います。

$$ \frac{| IP(B) - EA(B) |}{r_p(A)^2}, \frac{| r_s(A) - r_p(B) |}{\exp(r_s(A))}, \frac{| r_p(B) - r_s(B) |}{\exp(r_d(A))}$$


In [ ]:
def make_3var_desc(df, ndesc=3):
    """論文の３変数に変換する。

    Args:
        df (pd.DataFrame)): データ
        ndesc (int, optional): 変換後のデータ次元. Defaults to 3.

    Returns:
        [type]: [description]
    """
    desc1 = np.abs(df["IP_B"]-df["EA_B"]) / df["rp_A"]**2
    desc2 = np.abs(df["rs_A"]-df["rp_B"]) / np.exp(df["rs_A"])
    desc3 = np.abs(df["rp_B"]-df["rs_B"]) / np.exp(df["rd_A"])
    y = df["dE"]
    df2 = pd.DataFrame()
    if ndesc >= 1:
        df2["desc1"] = desc1
    if ndesc >= 2:
        df2["desc2"] = desc2
    if ndesc >= 3:
        df2["desc3"] = desc3
    df2["dE"] = y
    return df2


df_3var = make_3var_desc(df, 3)
df_3var.to_csv("../data_calculated/ZB_WZ_dE_3var.csv", index=False)
df_3var


In [ ]:
# 念の為もう一度読み込む。
df_3desc = pd.read_csv("../data_calculated/ZB_WZ_dE_3var.csv")
descriptor3_labels = ['desc1', 'desc2', 'desc3']
target3_label = 'dE'

In [ ]:
print("same?", np.all(df_3desc==df_3var))
df_diff = df_3desc-df_3var
print("similar?", np.all(np.abs(df_diff.values)<1e-10)) 
# 値がほぼ同じことを確認している。一度ファイルに保存しているので精度が落ちている。

In [ ]:
df_3desc.plot(x="desc1",y='dE',kind="scatter")
plt.show()

変換の確かめとして可視化を行います。論文にかかれていたとおり、第一説明変数が目的変数とほぼ比例するのが確認できました。

In [ ]:
import os
import seaborn as sns
from copy import deepcopy

imgfile = os.path.join(IMAGE_DIR, "ZBWZ3_pairplot.png")
if not os.path.isfile(imgfile):
    alllabels = deepcopy(descriptor3_labels)
    alllabels.append(target3_label)
    img = sns.pairplot(df_3desc[alllabels])
    os.makedirs(IMAGE_DIR, exist_ok=True)
    img.savefig(imgfile)
    
from IPython import display
display.Image(imgfile)

**参考文献**

1. Luca M. Ghiringhelli,
Jan Vybiral,
Sergey V. Levchenko,
Claudia Draxl,
and Matthias Scheffler,
"Big Data of Materials Science: Critical Role of the Descriptor",
Phys. Rev. Lett. 114, 105503 (2015)
